In [0]:
%run ../utils/adls_auth

In [0]:
# 2. Check row count
silver_df = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver")
print(f"Silver rows: {silver_df.count()}")


In [0]:
# 3. Check quarantine
quarantine_df = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_quarantine")
print(f"Quarantine rows: {quarantine_df.count()}")
# Expected: 0 (unless you injected bad data already)

In [0]:
# 4. Check dq_results
dq_df = spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/dq_results")
dq_df.groupBy("check_name", "severity").count().display()

In [0]:
# Check what's in quarantine
quarantine_df = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_quarantine")

# Sample quarantined rows
quarantine_df.display(10)


In [0]:
# Check which checks failed most
from pyspark.sql.functions import col, explode, size
# Show the failure reasons
quarantine_df.select(
    col("_dq_failures"),
    size(col("_dq_failures")).alias("failure_count")
).groupBy("failure_count").count().display()

In [0]:
# 1. Check if Weather Silver table exists
from delta.tables import DeltaTable
print(f"Weather Silver exists: {DeltaTable.isDeltaTable(spark, 'abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_silver')}")



In [0]:
# 2. Count rows
weather_df = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_silver")
print(f"Weather Silver rows: {weather_df.count():,}")



In [0]:
# 3. Check partitions
weather_df.groupBy("weather_date").count().orderBy("weather_date").display()


In [0]:

# 4. Check quarantine
weather_quarantine = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_quarantine")
print(f"Weather quarantine rows: {weather_quarantine.count()}")



In [0]:
# 5. Check DQ results for weather
from pyspark.sql.functions import sum
dq_df = spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/dq_results")

dq_df.filter("table_name = 'weather_silver'").groupBy("check_name", "severity").agg(
    sum("rows_failed").alias("total_failed")
).display()

In [0]:
# Full DQ results summary
dq_df = spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/dq_results")
dq_df.groupBy("table_name", "check_name", "severity").agg(
    sum("rows_failed").alias("total_failed")
).display()

In [0]:
from pyspark.sql.functions import col, count

# Check duplicate distribution
raw_df = spark.read.parquet("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_raw")

dupe_check = (
    raw_df.groupBy("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID")
    .agg(count("*").alias("n"))
    .filter("n > 1")
    .orderBy(col("n").desc())
)

print("Top 10 duplicate groups:")
dupe_check.show(10, truncate=False)

# Summary stats
dupe_check.select("n").describe().show()

In [0]:

checks_df = raw_df.select(
    col("tpep_pickup_datetime").isNull().alias("null_pickup_datetime"),
    col("tpep_dropoff_datetime").isNull().alias("null_dropoff_datetime"),
    col("PULocationID").isNull().alias("null_pickup_location"),
    (col("fare_amount") < 0).alias("negative_fare"),
    (col("trip_distance") < 0).alias("negative_distance"),
    (col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")).alias("pickup_after_dropoff"),
)

# Count failures per check
failure_counts = {}
for check in checks_df.columns:
    count = checks_df.filter(col(check) == True).count()
    failure_counts[check] = count
    print(f"{check}: {count:,}")

# Total rows
total_rows = raw_df.count()
print(f"\nTotal rows: {total_rows:,}")
print(f"Total critical failures (non-unique rows): {len(checks_df.filter(' OR '.join([f'{c} = True' for c in checks_df.columns])).collect()):,}")

In [0]:
negative_fares = raw_df.filter(col("fare_amount") < 0)
negative_fares.groupBy("payment_type", "VendorID").count().orderBy(col("count").desc()).show(10)

# Also check distribution by month
from pyspark.sql.functions import month, year
negative_fares.groupBy(year("tpep_pickup_datetime").alias("year"), month("tpep_pickup_datetime").alias("month")).count().orderBy("year", "month").display()

In [0]:
from pyspark.sql.functions import col, count, when

# Normalize negative fares against total trips by month for Vendor 2
vendor2_by_month = (
    raw_df.filter(col("VendorID") == 2)
    .groupBy("month")
    .agg(
        count("*").alias("total_trips"),
        count(when(col("fare_amount") < 0, True)).alias("negative_fare_trips")
    )
    .withColumn("negative_fare_rate", col("negative_fare_trips") / col("total_trips"))
    .orderBy("month")
)

vendor2_by_month.show(20)

In [0]:
# Check deletion vectors on all Silver tables (important for Synapse Serverless reads)
for t in ["trips_silver", "weather_silver", "trips_stream_silver"]:
    print(f"\n=== {t} ===")
    try:
        spark.sql(f"SHOW TBLPROPERTIES delta.`abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/{t}`").show(truncate=False)
    except Exception as e:
        print(f"Table {t} may not exist yet or error: {e}")